# Análise Exploratória de Dados - VariantClassifier

Este notebook realiza análise exploratória dos dados de variantes genômicas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Dados

In [ ]:
# Carregar dados de treino
train_df = pd.read_csv('../data/splits/train.csv')
val_df = pd.read_csv('../data/splits/val.csv')
test_df = pd.read_csv('../data/splits/test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Validation shape: {val_df.shape}')
print(f'Test shape: {test_df.shape}')

train_df.head()

## 2. Distribuição das Classes

In [ ]:
# Contagem de classes
class_counts = train_df['pathogenicity'].value_counts().sort_index()
class_percentages = (class_counts / len(train_df) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico de barras
class_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Contagem por Classe ACMG')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Contagem')
axes[0].tick_params(axis='x', rotation=45)

# Gráfico de pizza
axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribuição Percentual')

plt.tight_layout()
plt.show()

print('\nResumo de Classes:')
print(class_counts)
print('\nPercentuais:')
print(class_percentages)

## 3. Features Numéricas

In [ ]:
# Selecionar features numéricas
numeric_features = [
    'allele_frequency', 'allele_frequency_popmax', 'gnomad_homozygotes',
    'revel_score', 'cadd_phred', 'spliceai_delta', 'mutpred_score',
    'eigen_cadd', 'mvp_score', 'metasvm_score'
]

numeric_cols = [col for col in numeric_features if col in train_df.columns]
print(f'Features numéricas: {len(numeric_cols)}')

# Estatísticas descritivas
train_df[numeric_cols].describe().T

In [ ]:
# Distribuição de features numéricas por classe
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

key_features = ['allele_frequency', 'revel_score', 'cadd_phred', 'spliceai_delta']

for idx, feature in enumerate(key_features):
    if feature in train_df.columns:
        for class_label in train_df['pathogenicity'].unique():
            data = train_df[train_df['pathogenicity'] == class_label][feature]
            axes[idx].hist(data, alpha=0.5, label=class_label, bins=30)
        
        axes[idx].set_title(f'Distribuição: {feature}')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Frequência')
        axes[idx].legend()

plt.tight_layout()
plt.show()

## 4. Correlações

In [ ]:
# Matriz de correlação
correlation_matrix = train_df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1
)
plt.title('Matriz de Correlação - Features Numéricas')
plt.tight_layout()
plt.show()

In [ ]:
# Correlações mais fortes
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.3:  # Threshold de correlação moderada
            corr_pairs.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlação': corr_val
            })

if corr_pairs:
    corr_df = pd.DataFrame(corr_pairs).sort_values('Correlação', key=abs, ascending=False)
    print('Correlações mais fortes (|r| > 0.3):')
    display(corr_df)
else:
    print('Nenhuma correlação forte encontrada.')

## 5. Features Categóricas

In [ ]:
# Selecionar features categóricas
categorical_features = [
    'chromosome', 'gene', 'variant_type', 'ref', 'alt',
    'consequence', 'transcript'
]

categorical_cols = [col for col in categorical_features if col in train_df.columns]
print(f'Features categóricas: {len(categorical_cols)}')

# Análise de features categóricas
for col in categorical_cols:
    print(f'\n{col}:')
    print(f'  Valores únicos: {train_df[col].nunique()}')
    print(f'  Top 5 valores:')
    print(f'  {train_df[col].value_counts().head()}')

In [ ]:
# Distribuição de variant_type por pathogenicity
if 'variant_type' in train_df.columns:
    plt.figure(figsize=(12, 6))
    
    # Cross-tabulation
    ct = pd.crosstab(train_df['variant_type'], train_df['pathogenicity'])
    
    # Plot stacked bar
    ct.plot(kind='bar', stacked=True, figsize=(12, 6))
    plt.title('Distribuição de Variant Type por Pathogenicity')
    plt.xlabel('Variant Type')
    plt.ylabel('Contagem')
    plt.xticks(rotation=45)
    plt.legend(title='Pathogenicity')
    plt.tight_layout()
    plt.show()

## 6. Valores Ausentes

In [ ]:
# Análise de valores ausentes
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_pct
}).sort_values('Missing', ascending=False)

missing_df = missing_df[missing_df['Missing'] > 0]

if len(missing_df) > 0:
    print('Colunas com valores ausentes:')
    display(missing_df)
    
    # Visualização
    plt.figure(figsize=(12, 6))
    missing_df['Percentage'].plot(kind='bar')
    plt.title('Percentual de Valores Ausentes por Coluna')
    plt.xlabel('Coluna')
    plt.ylabel('Percentual (%)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum valor ausente encontrado.')

## 7. Análise por Gene

In [ ]:
# Top genes por frequência
if 'gene' in train_df.columns:
    top_genes = train_df['gene'].value_counts().head(20)
    
    plt.figure(figsize=(12, 6))
    top_genes.plot(kind='bar')
    plt.title('Top 20 Genes por Frequência')
    plt.xlabel('Gene')
    plt.ylabel('Contagem')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print('\nTop 20 genes:')
    print(top_genes)

In [ ]:
# Distribuição de patogenicidade por gene (top 10)
if 'gene' in train_df.columns:
    top_10_genes = train_df['gene'].value_counts().head(10).index.tolist()
    
    # Filtrar dados para top 10 genes
    top_10_df = train_df[train_df['gene'].isin(top_10_genes)]
    
    # Cross-tabulation
    ct = pd.crosstab(top_10_df['gene'], top_10_df['pathogenicity'])
    
    # Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')
    plt.title('Distribuição de Patogenicidade por Gene (Top 10)')
    plt.xlabel('Pathogenicity')
    plt.ylabel('Gene')
    plt.tight_layout()
    plt.show()

## 8. Resumo Estatístico

In [ ]:
print('\n' + '='*80)
print('RESUMO DA ANÁLISE EXPLORATÓRIA')
print('='*80)

print(f'\nTotal de variantes: {len(train_df)}')
print(f'Classes ACMG: {train_df["pathogenicity"].nunique()}')
print(f'Features numéricas: {len(numeric_cols)}')
print(f'Features categóricas: {len(categorical_cols)}')

print(f'\nDistribuição de classes:')
for cls, count in class_counts.items():
    print(f'  {cls}: {count} ({class_percentages[cls]}%)')

print(f'\nValores ausentes: {train_df.isnull().sum().sum()}')
print(f'Variáveis com valores ausentes: {len(missing_df) if len(missing_df) > 0 else 0}')